# Task 1 — Data Exploration & Enrichment
**Ethiopia Financial Inclusion Forecasting | Selam Analytics**

Objective: Understand the starter dataset schema, explore all record types, and enrich the dataset with additional observations, events, and impact links.

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
plt.style.use('seaborn-v0_8-whitegrid')
print("Imports OK")

## 2. Load Raw Dataset

In [ ]:
from src.data_loader import load_all
data = load_all()

main   = data['main']
obs    = data['observations']
events = data['events']
targets= data['targets']
impact = data['impact_links']
ref    = data['reference']

print(f"Main dataset : {main.shape}")
print(f"Observations : {obs.shape}")
print(f"Events       : {events.shape}")
print(f"Targets      : {targets.shape}")
print(f"Impact links : {impact.shape}")
print(f"Reference    : {ref.shape}")

## 3. Schema Overview

In [ ]:
print("=== Columns ===")
print(main.columns.tolist())
print("\n=== Record types ===")
print(main['record_type'].value_counts())
print("\n=== Pillars ===")
print(main['pillar'].value_counts(dropna=False))
print("\n=== Source types ===")
print(main['source_type'].value_counts(dropna=False))
print("\n=== Confidence levels ===")
print(main['confidence'].value_counts(dropna=False))

## 4. Observations — All Indicators

In [ ]:
obs_summary = (obs.groupby(['indicator_code','indicator','pillar'])
               .agg(records=('record_id','count'),
                    min_date=('observation_date','min'),
                    max_date=('observation_date','max'))
               .reset_index()
               .sort_values('pillar'))
print(obs_summary.to_string(index=False))

## 5. Events Catalog

In [ ]:
print(events[['record_id','category','indicator','observation_date','confidence']]
      .sort_values('observation_date')
      .to_string(index=False))

## 6. Impact Links

In [ ]:
imp_joined = (impact
    .merge(main[['record_id','indicator']], left_on='parent_id', right_on='record_id', suffixes=('','_event'))
    [['record_id','parent_id','indicator_event','pillar','related_indicator',
      'impact_direction','impact_magnitude','lag_months','evidence_basis']]
    .sort_values('parent_id'))
print(imp_joined.to_string(index=False))

## 7. Temporal Coverage Heatmap

In [ ]:
obs2 = obs.copy()
obs2['year'] = obs2['observation_date'].dt.year
coverage = obs2.groupby(['indicator_code','year']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(coverage, cmap='Blues', linewidths=0.5, ax=ax,
            cbar_kws={'label':'Record count'})
ax.set_title("Temporal Coverage — Records per Indicator per Year", fontsize=13, fontweight='bold')
ax.set_xlabel("Year")
ax.set_ylabel("Indicator Code")
plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/task1_temporal_coverage.png', dpi=150)
plt.show()
print("Saved: reports/figures/task1_temporal_coverage.png")

## 8. Run Enrichment

In [ ]:
from src.enrich_data import enrich, save_processed
enriched_main, enriched_impact = enrich(data['main'].copy(), data['impact_links'].copy())

print(f"Original records : {len(data['main'])}")
print(f"Enriched records : {len(enriched_main)}")
print(f"  Observations   : {(enriched_main['record_type']=='observation').sum()}")
print(f"  Events         : {(enriched_main['record_type']=='event').sum()}")
print(f"  Targets        : {(enriched_main['record_type']=='target').sum()}")
print(f"Impact links     : {len(enriched_impact)}")

save_processed(enriched_main, enriched_impact)

## 9. New Records Summary

In [ ]:
new_obs = enriched_main[
    (enriched_main['record_type']=='observation') &
    (~enriched_main['record_id'].isin(data['main']['record_id']))
][['record_id','indicator_code','value_numeric','observation_date','pillar','confidence','notes']]
print("=== New Observations ===")
print(new_obs.to_string(index=False))

new_evt = enriched_main[
    (enriched_main['record_type']=='event') &
    (~enriched_main['record_id'].isin(data['main']['record_id']))
][['record_id','category','indicator','observation_date','notes']]
print("\n=== New Events ===")
print(new_evt.to_string(index=False))

## 10. Schema Validation
Confirm all events have no pillar assigned (key design principle).

In [ ]:
events_enriched = enriched_main[enriched_main['record_type']=='event']
events_with_pillar = events_enriched['pillar'].dropna()
if len(events_with_pillar) == 0:
    print("✅ All events have pillar = NULL (schema compliant)")
else:
    print(f"⚠️  {len(events_with_pillar)} events have pillar set — review needed")

print("\nRecord ID uniqueness:", "✅ OK" if not enriched_main['record_id'].dropna().duplicated().any() else "❌ DUPLICATES FOUND")